### Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import os, pathlib
from pathlib import Path

In [ ]:
!pip install window_ops > /dev/null

In [ ]:
full_df = pd.read_parquet('full_exp_df.parquet')

Lag Features

In [ ]:
lags = ((np.arange(5) + 1).tolist() + (np.arange(5) + 46).tolist() +
        (np.arange(5) + (48 * 7) - 2).tolist() )
lags

[1, 2, 3, 4, 5, 46, 47, 48, 49, 50, 334, 335, 336, 337, 338]

In [ ]:
with LogTime():
    full_df, added_features = add_lags(
        full_df, lags = lags, column="energy_consumption", ts_id="LCLid", use_32_bit=True
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 17 seconds, 889 milliseconds and 121 microseconds
Features Created: energy_consumption_lag_1,energy_consumption_lag_2,energy_consumption_lag_3,energy_consumption_lag_4,energy_consumption_lag_5,energy_consumption_lag_46,energy_consumption_lag_47,energy_consumption_lag_48,energy_consumption_lag_49,energy_consumption_lag_50,energy_consumption_lag_334,energy_consumption_lag_335,energy_consumption_lag_336,energy_consumption_lag_337,energy_consumption_lag_338


Rolling

In [ ]:
with LogTime():
    full_df, added_features = add_rolling_features(
        full_df,
        rolls=[3, 6, 12, 48],
        column="energy_consumption",
        agg_funcs=["mean", "std"],
        ts_id="LCLid",
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 37 seconds, 838 milliseconds and 724 microseconds
Features Created: energy_consumption_rolling_3_mean,energy_consumption_rolling_3_std,energy_consumption_rolling_6_mean,energy_consumption_rolling_6_std,energy_consumption_rolling_12_mean,energy_consumption_rolling_12_std,energy_consumption_rolling_48_mean,energy_consumption_rolling_48_std


Seasonal Rolling

In [ ]:
with LogTime():
    full_df, added_features = add_seasonal_rolling_features(
        full_df,
        rolls=[3],
        seasonal_periods=[48, 48 * 7],
        column="energy_consumption",
        agg_funcs=["mean", "std"],
        ts_id="LCLid",
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 13 seconds, 486 milliseconds and 735 microseconds
Features Created: energy_consumption_48_seasonal_rolling_3_mean,energy_consumption_48_seasonal_rolling_3_std,energy_consumption_336_seasonal_rolling_3_mean,energy_consumption_336_seasonal_rolling_3_std


EWMA

In [ ]:
import math

t = np.arange(25).tolist()
plot_df = pd.DataFrame({"Timesteps behind t": t})
for alpha in [0.3, 0.5, 0.8]:
    weights = [alpha * math.pow((1 - alpha), i) for i in t]
    span = (2 - alpha) / alpha
    halflife = math.log(1 - alpha) / math.log(0.5)
    plot_df[f"Alpha={alpha} | Span={span:.2f}"] = weights

In [ ]:
fig = px.line(
    pd.melt(plot_df, id_vars="Timesteps behind t", var_name="Parameters"),
    x="Timesteps behind t",
    y="value",
    facet_col="Parameters",
)
fig.update_annotations(font=dict(size=16))
fig.show()

In [ ]:
with LogTime():
    # full_df, added_features = add_ewma(full_df, alphas=[0.2, 0.5, 0.9], column="energy_consumption", ts_id="LCLid", use_32_bit=True)
    full_df, added_features = add_ewma(
        full_df,
        spans=[48 * 60, 48 * 7, 48],
        column="energy_consumption",
        ts_id="LCLid",
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 4 seconds, 509 milliseconds and 969 microseconds
Features Created: energy_consumption_ewma_span_2880,energy_consumption_ewma_span_336,energy_consumption_ewma_span_48


Temporal Features

In [ ]:
with LogTime():
    full_df, added_features = add_temporal_features(
        full_df,
        field_name="timestamp",
        frequency="30min",
        add_elapsed=True,
        drop=False,
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 6 seconds, 240 milliseconds and 756 microseconds
Features Created: timestamp_Month,timestamp_Quarter,timestamp_Is_quarter_end,timestamp_Is_quarter_start,timestamp_Is_year_end,timestamp_Is_year_start,timestamp_Is_month_start,timestamp_Day,timestamp_Dayofweek,timestamp_Dayofyear,timestamp_Hour,timestamp_Minute,timestamp_Week,timestamp_Elapsed


Fourier Terms

In [ ]:
with LogTime():
    full_df, added_features = bulk_add_fourier_features(
        full_df,
        ["timestamp_Month", "timestamp_Hour", "timestamp_Minute"],
        max_values=[12, 24, 60],
        n_fourier_terms=5,
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 15 seconds, 630 milliseconds and 190 microseconds
Features Created: timestamp_Month_sin_1,timestamp_Month_sin_2,timestamp_Month_sin_3,timestamp_Month_sin_4,timestamp_Month_sin_5,timestamp_Month_cos_1,timestamp_Month_cos_2,timestamp_Month_cos_3,timestamp_Month_cos_4,timestamp_Month_cos_5,timestamp_Hour_sin_1,timestamp_Hour_sin_2,timestamp_Hour_sin_3,timestamp_Hour_sin_4,timestamp_Hour_sin_5,timestamp_Hour_cos_1,timestamp_Hour_cos_2,timestamp_Hour_cos_3,timestamp_Hour_cos_4,timestamp_Hour_cos_5,timestamp_Minute_sin_1,timestamp_Minute_sin_2,timestamp_Minute_sin_3,timestamp_Minute_sin_4,timestamp_Minute_sin_5,timestamp_Minute_cos_1,timestamp_Minute_cos_2,timestamp_Minute_cos_3,timestamp_Minute_cos_4,timestamp_Minute_cos_5


In [ ]:
full_df.columns

Index(['LCLid', 'timestamp', 'energy_consumption', 'timestamp_Week',
       'frequency', 'type', 'energy_consumption_lag_1',
       'energy_consumption_lag_2', 'energy_consumption_lag_3',
       'energy_consumption_lag_4', 'energy_consumption_lag_5',
       'energy_consumption_lag_46', 'energy_consumption_lag_47',
       'energy_consumption_lag_48', 'energy_consumption_lag_49',
       'energy_consumption_lag_50', 'energy_consumption_lag_334',
       'energy_consumption_lag_335', 'energy_consumption_lag_336',
       'energy_consumption_lag_337', 'energy_consumption_lag_338',
       'energy_consumption_rolling_3_mean', 'energy_consumption_rolling_3_std',
       'energy_consumption_rolling_6_mean', 'energy_consumption_rolling_6_std',
       'energy_consumption_rolling_12_mean',
       'energy_consumption_rolling_12_std',
       'energy_consumption_rolling_48_mean',
       'energy_consumption_rolling_48_std',
       'energy_consumption_48_seasonal_rolling_3_mean',
       'energy_consumptio

Plotting Fourier Terms

In [ ]:
plot_df = (
    full_df[["timestamp_Month", "timestamp_Month_sin_1"]]
    .drop_duplicates()
    .sort_values("timestamp_Month")
)
plot_df.columns = ["calendar", "fourier"]

plot_df = pd.concat([plot_df, plot_df, plot_df]).reset_index(drop=True)
# plot_df.reset_index(drop=True, inplace=True)

plot_df.reset_index(inplace=True)
plot_df["index"] += 1
plot_df = pd.melt(
    plot_df, id_vars="index", var_name="month", value_name="Representation"
)

In [ ]:
fig = px.line(plot_df, x="index", y="Representation", facet_row="month")
fig.update_layout(
    autosize=False,
    width=900,
    height=800,
    title_text="Step Function vs Continuous Function",
    title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
    titlefont={"size": 20},
    legend_title=None,

    xaxis=dict(
        title_text="Time",
    ),
)
fig.update_yaxes(matches=None)

In [ ]:
fig.update_xaxes(
    ticktext=np.arange(1, 13).tolist() * 3,
    tickvals=np.arange(len(plot_df)) + 1,
)
fig #.show()

Saving the feature engineered file

In [ ]:
full_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 10000000 entries, 0 to 4383
Columns: 79 entries, LCLid to timestamp_Minute_cos_5
dtypes: datetime64[ns](1), float32(60), float64(1), int32(14), object(3)
memory usage: 4.5 GB


In [ ]:
full_df[full_df["type"] == "train"].drop(columns="type").to_parquet("selected_blocks_train_feature_eng.parquet")

full_df[full_df["type"] == "val"].drop(columns="type").to_parquet("selected_blocks_val_feature_eng.parquet")

full_df[full_df["type"] == "test"].drop(columns="type").to_parquet("selected_blocks_test_eature_eng.parquet")

### Feature Engineering with Improvements

Features we are creating need the train and test dataset to be combined into a signle dataset with continuous time. In case of a production environment we create test period dataset with absent (zero) actual observations and continue.


In [ ]:
import collections
from collections import defaultdict

!pip install mlforecast > /dev/null
from mlforecast.lag_transforms import (
	RollingMean,
	RollingStd,
	RollingMin,
	RollingMax,
	SeasonalRollingMean,
	SeasonalRollingMin,
	SeasonalRollingMax,
	SeasonalRollingStd,
	ExponentiallyWeightedMean,
)

#### Lag Features

In [ ]:
lags = ((np.arange(5) + 1).tolist() + (np.arange(5) + 46).tolist() + (np.arange(5) + (48 * 7) - 2).tolist() )
lags

[1, 2, 3, 4, 5, 46, 47, 48, 49, 50, 334, 335, 336, 337, 338]

#### Rolling

All the other features apart from lags are added as LagTransforms in mlforecast, this due to a leakage in data (with rolling meand and current time step to be avoided). In mlforecast the transformations are added as a dictionary. To add transformations as {'1': [Transform for rolling Mean, Transform for Exponential Mean]} it will move back one time step and calculate the transformations.


In [ ]:
lag_transforms = defaultdict(list)

# Adding Rolling Mean, Rolling Std, with an offset of one timestep
lag_transforms[1]+= [
    RollingMean(window_size=n) for n in [3, 6, 12, 48]] + [RollingStd(window_size=n) for n in [3, 6, 12, 48] ]

#### Seasonal Rolling

This goes back seasonal length and calculates mean, max etc in a window from that starting point. Understand how to configure seasonal length and window size.

In [ ]:
''' Understand how to configure seasonal length and window size '''
from mlforecast.lag_transforms import SeasonalRollingMax

# generate sequential data to verify seasonal rolling
data = generate_daily_series(1,min_length=50, max_length=500, seed=42)
data['y'] = np.arange(1, len(data)+1)

season_length = 8
window_size=1

# Defining the Rolling window we want to test
seasonal_rolling_window = SeasonalRollingMax(
     window_size=window_size, season_length=season_length)

fcst = MLForecast(
    models=[],
    freq='D',
    lag_transforms={
    season_length: [seasonal_rolling_window],
    },
)
data_t = fcst.preprocess(data)
data_t.head()

In [ ]:
# Adding Seasonal Rolling Mean, Seasonal Rolling Std, with an offset of seasonal period timestep
lag_transforms[48]+= [SeasonalRollingMean(season_length=48, window_size=3)] + \
                     [SeasonalRollingStd(season_length=48, window_size=3)]

lag_transforms[48 * 7]+= [
    SeasonalRollingMean(season_length=48 * 7, window_size=3)] + [SeasonalRollingStd(season_length=48 * 7, window_size=3)]

#### EWMA

In [ ]:
import math

t = np.arange(25).tolist()
plot_df = pd.DataFrame({"Timesteps behind t": t})
for alpha in [0.3, 0.5, 0.8]:
    weights = [alpha * math.pow((1 - alpha), i) for i in t]
    span = (2 - alpha) / alpha
    halflife = math.log(1 - alpha) / math.log(0.5)
    plot_df[f"Alpha={alpha} | Span={span:.2f}"] = weights

In [ ]:
fig = px.line(
    pd.melt(plot_df, id_vars="Timesteps behind t", var_name="Parameters"),
    x="Timesteps behind t",
    y="value",
    facet_col="Parameters",
)
fig.update_layout(
    autosize=False,
    width=1200,
    height=500,
    yaxis=dict(
        title_text="Weights",
        titlefont=dict(size=15),
        tickfont=dict(size=15),
    ),
    xaxis=dict(
        titlefont=dict(size=15),
        tickfont=dict(size=15),
    ),
)
fig.update_annotations(font=dict(size=16))
fig.show()

In [ ]:
# Adding Rolling Mean, Rolling Std, with an offset of one timestep
lag_transforms[1] += [
    ExponentiallyWeightedMean(alpha=alpha) for alpha in [0.2, 0.5, 0.9]]

#### Temporal Features

In [ ]:
# Define the features you need in the model
# these should either be strings (pandas date function) or functions that take date as an argument
temporal_features = [
    "month", "quarter", "is_quarter_end", "is_quarter_start", "is_year_end", "is_year_start", "is_month_start", "is_month_end", "week", "day", "dayofweek", "dayofyear", "hour", "minute", ]

In [ ]:
with LogTime():
    full_df, added_features = add_temporal_features(
        full_df,
        field_name="timestamp",
        frequency="30min",
        add_elapsed=True,
        drop=False,
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 0.56 seconds
Features Created: timestamp_Month,timestamp_Quarter,timestamp_Is_quarter_end,timestamp_Is_quarter_start,timestamp_Is_year_end,timestamp_Is_year_start,timestamp_Is_month_start,timestamp_Day,timestamp_Dayofweek,timestamp_Dayofyear,timestamp_Hour,timestamp_Minute,timestamp_Week,timestamp_Elapsed


#### Calculating the Features

In [ ]:
from mlforecast import MLForecast

In [ ]:
import pandas as pd

fcst = MLForecast(
		models=[],
		freq='30min', # Defining the frequency of the data
		lags=lags, # Defining the Lags we need to create
		# Defining some transformations we need to do to the lags (offsets)
		lag_transforms=lag_transforms,
		date_features=temporal_features # Defining the date features we need
)
with LogTime():
	# Drop rows with null values in the 'energy_consumption' column before preprocessing
	full_df = full_df.dropna(subset=['energy_consumption'])

	# Explicitly handle missing timestamps by reindexing each series
	# Create a complete date range for each series and propagate LCLid and fill other columns
	full_df_processed = full_df.groupby('LCLid', group_keys=False).apply(lambda x:
		x.set_index('timestamp')
		.reindex(pd.date_range(
			start=x['timestamp'].min(),
			end=x['timestamp'].max(),
			freq='30min'
		))
		.assign(LCLid=x['LCLid'].iloc[0]) # Propagate LCLid to new rows
		.pipe(lambda df: df.assign(frequency=df['frequency'].ffill().bfill())) # Fill frequency
		.pipe(lambda df: df.assign(type=df['type'].ffill().bfill())) # Fill type
		.pipe(lambda df: df.assign(energy_consumption=df['energy_consumption'].fillna(0))) # Fill energy_consumption with 0
		.reset_index(names=['timestamp'])
	).reset_index(drop=True)

	# Drop the 'type' column as it is not a feature for mlforecast and can cause issues if considered static.
	# It is used for splitting data, not as a model feature.
	full_df_processed = full_df_processed.drop(columns=['type'])

	full_df = fcst.preprocess(
		full_df_processed,
		time_col="timestamp",
		id_col="LCLid",
		target_col="energy_consumption"
)

In [ ]:
full_df.columns

Index(['LCLid', 'timestamp', 'energy_consumption', 'timestamp_Week',
       'frequency', 'type', 'timestamp_Month', 'timestamp_Quarter',
       'timestamp_Is_quarter_end', 'timestamp_Is_quarter_start',
       'timestamp_Is_year_end', 'timestamp_Is_year_start',
       'timestamp_Is_month_start', 'timestamp_Day', 'timestamp_Dayofweek',
       'timestamp_Dayofyear', 'timestamp_Hour', 'timestamp_Minute',
       'timestamp_Elapsed'],
      dtype='object')

#### Fourier Terms

In [ ]:
full_df.columns

In [ ]:
with LogTime():
    full_df, added_features = bulk_add_fourier_features(
        full_df,
        ["timestamp_Month", "timestamp_Hour", "timestamp_Minute"],
        max_values=[12, 24, 60],
        n_fourier_terms=5,
        use_32_bit=True,
    )
print(f"Features Created: {','.join(added_features)}")

Time Elapsed: 3.57 seconds
Features Created: timestamp_Month_sin_1,timestamp_Month_sin_2,timestamp_Month_sin_3,timestamp_Month_sin_4,timestamp_Month_sin_5,timestamp_Month_cos_1,timestamp_Month_cos_2,timestamp_Month_cos_3,timestamp_Month_cos_4,timestamp_Month_cos_5,timestamp_Hour_sin_1,timestamp_Hour_sin_2,timestamp_Hour_sin_3,timestamp_Hour_sin_4,timestamp_Hour_sin_5,timestamp_Hour_cos_1,timestamp_Hour_cos_2,timestamp_Hour_cos_3,timestamp_Hour_cos_4,timestamp_Hour_cos_5,timestamp_Minute_sin_1,timestamp_Minute_sin_2,timestamp_Minute_sin_3,timestamp_Minute_sin_4,timestamp_Minute_sin_5,timestamp_Minute_cos_1,timestamp_Minute_cos_2,timestamp_Minute_cos_3,timestamp_Minute_cos_4,timestamp_Minute_cos_5


#### Plotting Fourier Terms

In [ ]:
plot_df = (
    full_df[["timestamp_Month", "timestamp_Month_sin_1"]]
    .drop_duplicates()
    .sort_values("timestamp_Month")
)
plot_df.columns = ["calendar", "fourier"]

plot_df = pd.concat([plot_df, plot_df, plot_df]).reset_index(drop=True)
# plot_df.reset_index(drop=True, inplace=True)

plot_df.reset_index(inplace=True)
plot_df["index"] += 1
plot_df = pd.melt(
    plot_df, id_vars="index", var_name="month", value_name="Representation"
)

In [ ]:
fig = px.line(plot_df, x="index", y="Representation", facet_row="month")
fig.update_layout(
    autosize=False,
    width=900,
    height=800,
    title_text="Step Function vs Continuous Function",
    title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
    titlefont={"size": 20},
    legend_title=None,

    xaxis=dict(
        title_text="Time",
    ),
)
fig.update_yaxes(matches=None)

In [ ]:
fig.update_xaxes(
    ticktext=np.arange(1, 13).tolist() * 3,
    tickvals=np.arange(len(plot_df)) + 1,
)
fig #.show()

Saving the feature engineered file

In [ ]:
full_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 960853 entries, 0 to 2943
Columns: 49 entries, LCLid to timestamp_Minute_cos_5
dtypes: datetime64[ns](1), float32(30), float64(1), int32(14), object(3)
memory usage: 335.3 MB
